In [20]:
!pip install lifelines

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceError

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

In [21]:
#1.데이터로드
train = pd.read_csv('train.csv') 
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')
print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(train.head())

Train shape: (221, 37)
Test shape : (95, 35)
   event_id  num_perimeters_0_5h  dt_first_last_0_5h  \
0  10892457                    3            4.265188   
1  11757157                    2            1.169918   
2  11945086                    4            4.777526   
3  12044083                    1            0.000000   
4  12052347                    2            4.975273   

   low_temporal_resolution_0_5h  area_first_ha  area_growth_abs_0_5h  \
0                             0      79.696304              2.875935   
1                             0       8.946749              0.000000   
2                             0     106.482638              0.000000   
3                             1      67.631125              0.000000   
4                             0      35.632874              0.000000   

   area_growth_rel_0_5h  area_growth_rate_ha_per_h  log1p_area_first  \
0              0.036086                   0.674281          4.390693   
1              0.000000                  

In [22]:
# 기본 컬럼 설정
ID_COL = "event_id"
TIME_COL = "time_to_hit_hours"
EVENT_COL = "event"

HORIZONS = [12, 24, 48, 72]
N_FOLDS = 5
RANDOM_STATE = 42

drop_cols = [ID_COL, TIME_COL, EVENT_COL]
base_feature_cols = [c for c in train.columns if c not in drop_cols]

X_raw = train[base_feature_cols].copy()
X_test_raw = test[base_feature_cols].copy()

y_time = train[TIME_COL].copy()
y_event = train[EVENT_COL].copy()

print("num raw features:", len(base_feature_cols))
print(base_feature_cols)

num raw features: 34
['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month']


In [23]:
# ── log1p 변환: dist_min_ci_0_5h → log1p_dist_min_ci_0_5h ──────────────
# dist_min_ci_0_5h는 일부 샘플에서 완전분리(complete separation)를 일으켜
# CoxPH 수렴 경고의 원인이 됩니다. log1p 변환으로 값 범위를 압축합니다.

LOG1P_SRC = "dist_min_ci_0_5h"
LOG1P_DST = "log1p_dist_min_ci_0_5h"

X_raw[LOG1P_DST] = np.log1p(X_raw[LOG1P_SRC])
X_raw = X_raw.drop(columns=[LOG1P_SRC])

X_test_raw[LOG1P_DST] = np.log1p(X_test_raw[LOG1P_SRC])
X_test_raw = X_test_raw.drop(columns=[LOG1P_SRC])

print(f"'{LOG1P_SRC}' → '{LOG1P_DST}' 변환 완료")
print(f"변환 후 피처 수: {len(X_raw.columns)}")
print(f"\n[log1p 변환된 컬럼 통계]")
print(X_raw[LOG1P_DST].describe())

'dist_min_ci_0_5h' → 'log1p_dist_min_ci_0_5h' 변환 완료
변환 후 피처 수: 34

[log1p 변환된 컬럼 통계]
count    221.000000
mean      10.179107
std        2.138115
min        5.729952
25%        7.995043
50%       10.365950
75%       12.207221
max       13.538045
Name: log1p_dist_min_ci_0_5h, dtype: float64


In [24]:
# fold 내부 전처리 함수
def fit_preprocess(X_tr_raw, X_va_raw=None, X_te_raw=None, corr_threshold=0.95):
    X_tr = X_tr_raw.copy().replace([np.inf, -np.inf], np.nan)
    X_va = None if X_va_raw is None else X_va_raw.copy().replace([np.inf, -np.inf], np.nan)
    X_te = None if X_te_raw is None else X_te_raw.copy().replace([np.inf, -np.inf], np.nan)

    # 1) train fold 기준 median imputation
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    if X_va is not None:
        X_va = X_va.fillna(medians)
    if X_te is not None:
        X_te = X_te.fillna(medians)

    # 2) high-corr drop도 train fold 기준으로만 결정
    corr_matrix = X_tr.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > corr_threshold)]

    # 핵심 변수 보호
    protected_cols = {"log1p_dist_min_ci_0_5h"}
    high_corr_cols = [c for c in high_corr_cols if c not in protected_cols]

    X_tr = X_tr.drop(columns=high_corr_cols, errors="ignore")
    if X_va is not None:
        X_va = X_va.drop(columns=high_corr_cols, errors="ignore")
    if X_te is not None:
        X_te = X_te.drop(columns=high_corr_cols, errors="ignore")

    # 3) low variance drop
    low_var_cols = []
    for col in X_tr.columns:
        if X_tr[col].nunique(dropna=False) <= 1:
            low_var_cols.append(col)
        elif X_tr[col].var() < 1e-8:
            low_var_cols.append(col)

    X_tr = X_tr.drop(columns=low_var_cols, errors="ignore")
    if X_va is not None:
        X_va = X_va.drop(columns=low_var_cols, errors="ignore")
    if X_te is not None:
        X_te = X_te.drop(columns=low_var_cols, errors="ignore")

    # 4) scaling
    feature_cols = X_tr.columns.tolist()
    scaler = StandardScaler()

    X_tr = pd.DataFrame(
        scaler.fit_transform(X_tr),
        columns=feature_cols,
        index=X_tr.index
    )

    if X_va is not None:
        X_va = pd.DataFrame(
            scaler.transform(X_va[feature_cols]),
            columns=feature_cols,
            index=X_va.index
        )

    if X_te is not None:
        X_te = pd.DataFrame(
            scaler.transform(X_te[feature_cols]),
            columns=feature_cols,
            index=X_te.index
        )

    prep = {
        "medians": medians,
        "high_corr_cols": high_corr_cols,
        "low_var_cols": low_var_cols,
        "feature_cols": feature_cols,
        "scaler": scaler
    }

    return X_tr, X_va, X_te, prep


def apply_preprocess(X_raw_input, prep):
    X = X_raw_input.copy().replace([np.inf, -np.inf], np.nan)
    X = X.fillna(prep["medians"])
    X = X.drop(columns=prep["high_corr_cols"], errors="ignore")
    X = X.drop(columns=prep["low_var_cols"], errors="ignore")
    X = X[prep["feature_cols"]]

    X = pd.DataFrame(
        prep["scaler"].transform(X),
        columns=prep["feature_cols"],
        index=X.index
    )
    return X

print("preprocess helpers ready")

preprocess helpers ready


In [25]:
# Brier함수 정의
def brier_score_censored_comp(times, events, pred_event_prob, H):
    """
    competition rule:
    - event=1 and time<=H  -> label=1, include
    - event=1 and time>H   -> label=0, include
    - event=0 and time>=H  -> label=0, include
    - event=0 and time<H   -> censored before horizon, exclude
    """
    times = np.asarray(times)
    events = np.asarray(events)
    pred_event_prob = np.asarray(pred_event_prob)

    include_mask = (events == 1) | ((events == 0) & (times >= H))
    y_true = ((events == 1) & (times <= H)).astype(float)

    y_true = y_true[include_mask]
    y_pred = pred_event_prob[include_mask]

    if len(y_true) == 0:
        return np.nan, 0

    bs = np.mean((y_true - y_pred) ** 2)
    return bs, len(y_true)

In [26]:
# CV 함수 정의
def run_lasso_cox_cv(penalizer):
    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof_risk = np.zeros(len(X_raw))
    oof_prob = np.zeros((len(X_raw), len(HORIZONS)))
    fold_cindex = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_raw, y_event), 1):
        print(f"\n--- fold {fold} / penalizer={penalizer} ---")

        X_tr_raw = X_raw.iloc[tr_idx].copy()
        X_va_raw = X_raw.iloc[va_idx].copy()

        X_tr, X_va, _, prep = fit_preprocess(
            X_tr_raw,
            X_va_raw=X_va_raw,
            X_te_raw=None,
            corr_threshold=0.95
        )

        print("features after preprocess:", len(prep["feature_cols"]))

        df_tr = X_tr.copy()
        df_tr[TIME_COL] = y_time.iloc[tr_idx].values
        df_tr[EVENT_COL] = y_event.iloc[tr_idx].values

        cph = CoxPHFitter(penalizer=penalizer, l1_ratio=1.0)

        try:
            cph.fit(
                df_tr,
                duration_col=TIME_COL,
                event_col=EVENT_COL,
                show_progress=False
            )
        except ConvergenceError:
            print(f"ConvergenceError at fold {fold}, penalizer={penalizer}")
            return None
        except Exception as e:
            print(f"Error at fold {fold}, penalizer={penalizer}: {e}")
            return None

        risk_va = cph.predict_partial_hazard(X_va).values.reshape(-1)
        oof_risk[va_idx] = risk_va

        cidx = concordance_index(
            y_time.iloc[va_idx],
            -risk_va,
            y_event.iloc[va_idx]
        )
        fold_cindex.append(cidx)
        print("fold c-index:", round(cidx, 6))

        surv_va = cph.predict_survival_function(X_va)

        for h_i, H in enumerate(HORIZONS):
            t = min(H, float(surv_va.index.max()))
            surv_at_t = surv_va.asof(t)
            event_prob = 1.0 - surv_at_t.values
            oof_prob[va_idx, h_i] = np.clip(event_prob, 0, 1)

    # 수치오차 방지용 monotonicity 보정
    oof_prob = np.maximum.accumulate(oof_prob, axis=1)

    oof_cindex = concordance_index(y_time, -oof_risk, y_event)

    brier_24, n24 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 1], 24)
    brier_48, n48 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 2], 48)
    brier_72, n72 = brier_score_censored_comp(y_time, y_event, oof_prob[:, 3], 72)

    weighted_brier = 0.3 * brier_24 + 0.4 * brier_48 + 0.3 * brier_72
    hybrid_score = 0.3 * oof_cindex + 0.7 * (1 - weighted_brier)

    return {
        "penalizer": penalizer,
        "fold_cindex_mean": float(np.mean(fold_cindex)),
        "fold_cindex_std": float(np.std(fold_cindex)),
        "oof_cindex": float(oof_cindex),
        "brier_24": float(brier_24),
        "brier_48": float(brier_48),
        "brier_72": float(brier_72),
        "weighted_brier": float(weighted_brier),
        "hybrid_score": float(hybrid_score),
        "n24": int(n24),
        "n48": int(n48),
        "n72": int(n72),
        "oof_risk": oof_risk,
        "oof_prob": oof_prob
    }

In [27]:
# penalizer 후보 실험
penalizer_grid = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08]

results = []

for p in penalizer_grid:
    print(f"\n================ penalizer={p} ================\n")
    res = run_lasso_cox_cv(p)

    if res is None:
        print(f"penalizer={p} failed\n")
        continue

    results.append(res)

    print("\n[summary]")
    print("fold mean c-index :", round(res["fold_cindex_mean"], 6))
    print("oof c-index       :", round(res["oof_cindex"], 6))
    print("brier@24          :", round(res["brier_24"], 6), f"(n={res['n24']})")
    print("brier@48          :", round(res["brier_48"], 6), f"(n={res['n48']})")
    print("brier@72          :", round(res["brier_72"], 6), f"(n={res['n72']})")
    print("weighted brier    :", round(res["weighted_brier"], 6))
    print("hybrid score      :", round(res["hybrid_score"], 6))


================ penalizer=0.02 ================


--- fold 1 / penalizer=0.02 ---
features after preprocess: 22
fold c-index: 0.864971

--- fold 2 / penalizer=0.02 ---
features after preprocess: 25
fold c-index: 0.923767

--- fold 3 / penalizer=0.02 ---
features after preprocess: 25
fold c-index: 0.901709

--- fold 4 / penalizer=0.02 ---
features after preprocess: 25
fold c-index: 0.955375

--- fold 5 / penalizer=0.02 ---
features after preprocess: 23
fold c-index: 0.922432

[summary]
fold mean c-index : 0.913651
oof c-index       : 0.907585
brier@24          : 0.059877 (n=196)
brier@48          : 0.051994 (n=166)
brier@72          : 0.032907 (n=69)
weighted brier    : 0.048633
hybrid score      : 0.938233

================ penalizer=0.03 ================


--- fold 1 / penalizer=0.03 ---
features after preprocess: 22
fold c-index: 0.933464

--- fold 2 / penalizer=0.03 ---
features after preprocess: 25
fold c-index: 0.919283

--- fold 3 / penalizer=0.03 ---
features after preprocess:

c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
  warnings.warn(


fold c-index: 0.930818

[summary]
fold mean c-index : 0.926123
oof c-index       : 0.923485
brier@24          : 0.057059 (n=196)
brier@48          : 0.047115 (n=166)
brier@72          : 0.015977 (n=69)
weighted brier    : 0.040757
hybrid score      : 0.948516

================ penalizer=0.04 ================


--- fold 1 / penalizer=0.04 ---
features after preprocess: 22
fold c-index: 0.939335

--- fold 2 / penalizer=0.04 ---
features after preprocess: 25
fold c-index: 0.91704

--- fold 3 / penalizer=0.04 ---
features after preprocess: 25
fold c-index: 0.903846

--- fold 4 / penalizer=0.04 ---
features after preprocess: 25
fold c-index: 0.955375

--- fold 5 / penalizer=0.04 ---
features after preprocess: 23
fold c-index: 0.926625

[summary]
fold mean c-index : 0.928444
oof c-index       : 0.925141
brier@24          : 0.052456 (n=196)
brier@48          : 0.044078 (n=166)
brier@72          : 0.015062 (n=69)
weighted brier    : 0.037887
hybrid score      : 0.951022

================ penal

c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
  warnings.warn(


fold c-index: 0.935421

--- fold 2 / penalizer=0.06 ---
features after preprocess: 25
fold c-index: 0.919283

--- fold 3 / penalizer=0.06 ---
features after preprocess: 25
fold c-index: 0.903846

--- fold 4 / penalizer=0.06 ---
features after preprocess: 25
ConvergenceError at fold 4, penalizer=0.06
penalizer=0.06 failed


================ penalizer=0.08 ================


--- fold 1 / penalizer=0.08 ---
features after preprocess: 22


c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1679: RuntimeWarning: overflow encountered in exp
  scores = weights * exp(dot(X, beta))
c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1731: RuntimeWarning: divide by zero encountered in divide
  denom = 1.0 / np.array([risk_phi])
c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1733: RuntimeWarning: invalid value encountered in multiply
  a1 = risk_phi_x_x * denom
c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1735: RuntimeWarning: invalid value encountered in multiply
  summand = numer * denom[:, None]
c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1740: RuntimeWarning: divide by zero encountered in log
  log_lik = log_lik + dot(x_death_sum, be

fold c-index: 0.941292

--- fold 2 / penalizer=0.08 ---
features after preprocess: 25
fold c-index: 0.923767

--- fold 3 / penalizer=0.08 ---
features after preprocess: 25
fold c-index: 0.897436

--- fold 4 / penalizer=0.08 ---
features after preprocess: 25


c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
  warnings.warn(
c:\Users\xotjs\AppData\Local\Programs\Python\Python312\Lib\site-packages\lifelines\fitters\coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-model
  warnings.warn(


fold c-index: 0.880325

--- fold 5 / penalizer=0.08 ---
features after preprocess: 23
fold c-index: 0.93501

[summary]
fold mean c-index : 0.915566
oof c-index       : 0.905515
brier@24          : 0.069437 (n=196)
brier@48          : 0.064805 (n=166)
brier@72          : 0.052869 (n=69)
weighted brier    : 0.062614
hybrid score      : 0.927825


**Hybrid score가 0.04의 경우에 가장 최고성능을 보임**

In [28]:
# 결과 표 정리
result_df = pd.DataFrame([{
    "penalizer": r["penalizer"],
    "fold_cindex_mean": r["fold_cindex_mean"],
    "fold_cindex_std": r["fold_cindex_std"],
    "oof_cindex": r["oof_cindex"],
    "brier_24": r["brier_24"],
    "brier_48": r["brier_48"],
    "brier_72": r["brier_72"],
    "weighted_brier": r["weighted_brier"],
    "hybrid_score": r["hybrid_score"],
} for r in results])

result_df = result_df.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
result_df

,penalizer,fold_cindex_mean,fold_cindex_std,oof_cindex,brier_24,brier_48,brier_72,weighted_brier,hybrid_score
0,0.04,0.928444,0.017789,0.925141,0.052456,0.044078,0.015062,0.037887,0.951022
1,0.03,0.926123,0.013496,0.923485,0.057059,0.047115,0.015977,0.040757,0.948516
2,0.02,0.913651,0.029775,0.907585,0.059877,0.051994,0.032907,0.048633,0.938233
3,0.05,0.915054,0.030556,0.911229,0.070536,0.060941,0.015935,0.050318,0.938146
4,0.08,0.915566,0.023142,0.905515,0.069437,0.064805,0.052869,0.062614,0.927825


In [29]:
# best penalizer 선택
best_penalizer = result_df.loc[0, "penalizer"]
print("best_penalizer:", best_penalizer)

best_res = [r for r in results if r["penalizer"] == best_penalizer][0]

best_penalizer: 0.04


In [30]:
# OOF 예측분포확인
oof_prob = best_res["oof_prob"]

oof_check = pd.DataFrame({
    "event_id": train[ID_COL],
    "prob_12h": oof_prob[:, 0],
    "prob_24h": oof_prob[:, 1],
    "prob_48h": oof_prob[:, 2],
    "prob_72h": oof_prob[:, 3],
    "time_to_hit_hours": y_time,
    "event": y_event
})

print(oof_check.head())

for col in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
    print(col, "min=", oof_check[col].min(), "max=", oof_check[col].max(), "std=", oof_check[col].std())

   event_id  prob_12h  prob_24h  prob_48h  prob_72h  time_to_hit_hours  event
0  10892457  0.646747  0.858565  0.905877  0.928330          18.892512      0
1  11757157  0.776210  0.940727  0.960100  0.998548          22.048108      1
2  11945086  0.675558  0.900476  0.935510  0.994267           0.888895      1
3  12044083  0.019010  0.035577  0.040458  0.080363          60.953021      0
4  12052347  0.178429  0.304942  0.361463  0.581437          44.990274      0
prob_12h min= 0.0015154771572863934 max= 1.0 std= 0.3046979372476993
prob_24h min= 0.002846671423801017 max= 1.0 std= 0.3657743121793655
prob_48h min= 0.0034383589724100716 max= 1.0 std= 0.3794250270635925
prob_72h min= 0.003834119266443059 max= 1.0 std= 0.4074938776840771


In [31]:
for col in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
    print(col, "count == 1.0 :", (oof_check[col] >= 0.999999).sum())
    print(col, "count == 0.0 :", (oof_check[col] <= 0.000001).sum())

prob_12h count == 1.0 : 2
prob_12h count == 0.0 : 0
prob_24h count == 1.0 : 2
prob_24h count == 0.0 : 0
prob_48h count == 1.0 : 4
prob_48h count == 0.0 : 0
prob_72h count == 1.0 : 7
prob_72h count == 0.0 : 0


In [32]:
# 전체 train 기준 전처리 학습
X_full, _, X_test_final, full_prep = fit_preprocess(
    X_raw,
    X_va_raw=None,
    X_te_raw=X_test_raw,
    corr_threshold=0.95
)

print("final feature count:", len(full_prep["feature_cols"]))
print(full_prep["feature_cols"])

final feature count: 25
['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'radial_growth_m', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'log1p_dist_min_ci_0_5h']


In [33]:
# 전체 trainset으로 최종 학습
df_full = X_full.copy()
df_full[TIME_COL] = y_time.values
df_full[EVENT_COL] = y_event.values

final_cph = CoxPHFitter(penalizer=float(best_penalizer), l1_ratio=1.0)
final_cph.fit(
    df_full,
    duration_col=TIME_COL,
    event_col=EVENT_COL,
    show_progress=False
)

print("final model fitted")

final model fitted


In [34]:
# 살아남은 계수 보기
coef_df = final_cph.params_.reset_index()
coef_df.columns = ["feature", "coef"]
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

print(coef_df.head(20).to_string(index=False))

nonzero_df = coef_df[coef_df["abs_coef"] > 1e-6]
print("\nnonzero coef count:", len(nonzero_df))

                     feature          coef     abs_coef
      log1p_dist_min_ci_0_5h -2.154955e+00 2.154955e+00
               alignment_abs  2.025477e-01 2.025477e-01
       cross_track_component -1.924918e-01 1.924918e-01
low_temporal_resolution_0_5h -1.825938e-01 1.825938e-01
                log1p_growth  9.121732e-02 9.121732e-02
               alignment_cos -5.471109e-02 5.471109e-02
          spread_bearing_sin  5.363186e-02 5.363186e-02
            dist_fit_r2_0_5h  4.484474e-02 4.484474e-02
          dt_first_last_0_5h  3.004705e-07 3.004705e-07
         num_perimeters_0_5h  2.658175e-07 2.658175e-07
           event_start_month  1.300774e-07 1.300774e-07
          spread_bearing_cos -1.113337e-07 1.113337e-07
           along_track_speed -1.067380e-07 1.067380e-07
       event_start_dayofweek  7.105919e-08 7.105919e-08
          spread_bearing_deg  6.922061e-08 6.922061e-08
            log1p_area_first  5.278115e-08 5.278115e-08
            dist_std_ci_0_5h  4.269095e-08 4.269

In [35]:
# test 예측
test_survival = final_cph.predict_survival_function(X_test_final)

submission = pd.DataFrame({ID_COL: test[ID_COL].values})

for H in HORIZONS:
    t = min(H, float(test_survival.index.max()))
    surv_at_t = test_survival.asof(t)
    submission[f"prob_{H}h"] = 1.0 - surv_at_t.values

prob_cols = [f"prob_{H}h" for H in HORIZONS]

for col in prob_cols:
    submission[col] = submission[col].clip(0, 1)

submission[prob_cols] = np.maximum.accumulate(submission[prob_cols].values, axis=1)

submission.head()

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.010412,0.019786,0.023727,0.045265
1,13353600,0.372190,0.588842,0.656294,0.872563
2,13942327,0.052065,0.097046,0.115441,0.210712
3,16112781,0.411773,0.636913,0.703993,0.904472
4,17132808,0.116456,0.210527,0.247270,0.421858


In [36]:
# 제출 전 체크
for col in prob_cols:
    print(col, "min=", submission[col].min(), "max=", submission[col].max(), "std=", submission[col].std())

mono_bad = submission[
    (submission["prob_12h"] > submission["prob_24h"]) |
    (submission["prob_24h"] > submission["prob_48h"]) |
    (submission["prob_48h"] > submission["prob_72h"])
]

print("monotonicity violations:", len(mono_bad))
print("submission shape:", submission.shape)

# 같은 예측값이 얼마나 줄었는지 확인
dup_count = submission[prob_cols].round(6).duplicated().sum()
print("duplicated prediction rows (rounded 6):", dup_count)

print("\nTop duplicated probability vectors:")
print(submission[prob_cols].round(6).value_counts().head(10))

prob_12h min= 0.0015851583519607138 max= 0.9999999999999998 std= 0.30459646337731533
prob_24h min= 0.003024239666205908 max= 1.0 std= 0.35481571847147103
prob_48h min= 0.0036328584127980346 max= 1.0 std= 0.3667728890996175
prob_72h min= 0.006996025448492982 max= 1.0 std= 0.39961788198176523
monotonicity violations: 0
submission shape: (95, 5)
duplicated prediction rows (rounded 6): 0

Top duplicated probability vectors:
prob_12h  prob_24h  prob_48h  prob_72h
0.001585  0.003024  0.003633  0.006996    1
0.002167  0.004134  0.004965  0.009556    1
0.002205  0.004206  0.005052  0.009723    1
0.002705  0.005159  0.006196  0.011917    1
0.003109  0.005927  0.007118  0.013685    1
0.003155  0.006015  0.007224  0.013888    1
0.003226  0.006149  0.007385  0.014196    1
0.003831  0.007302  0.008768  0.016844    1
0.003838  0.007315  0.008784  0.016875    1
0.003988  0.007600  0.009125  0.017527    1
Name: count, dtype: int64


In [37]:
submission.to_csv("lasso_cox_baseline_submission.csv", index=False)
print("saved: lasso_cox_baseline_submission.csv")

saved: lasso_cox_baseline_submission.csv


In [38]:
np.save("cox_oof_prob.npy", best_res["oof_prob"])
np.save("cox_oof_risk.npy", best_res["oof_risk"])

**GPT평가: Lasso-Cox 베이스라인은 생존분석 구조에 맞게 구현되었으며, 현재 모델은 12h, 24h, 48h, 72h 확률 예측의 단조성을 만족하고 OOF 기준으로도 ranking과 calibration을 모두 일정 수준 확보해 baseline으로서 비교 가능한 성능을 보였습니다. 특히 거리 및 초기 확산 관련 변수가 주요 예측 축으로 작동한다는 점에서 문제 구조와의 정합성도 확인되었습니다. 다만 일부 변수(dist_min_ci_0_5h)의 강한 분리력으로 인해 convergence warning이 남아 있어 수치적으로 완전히 안정적인 최종 모델이라기보다는, 이후 RSF 및 추가 튜닝과 비교하기 위한 strong baseline으로 해석하는 것이 적절합니다.**

++ dist_min_ci_0_5h는 크고 작은 극단값이 많음 --> log1p로 로그변환해 큰 값들 압축, 극단값 완화